# Training Results Viewer (macro evaluation)

This notebook allows you to load and visualize the results of finetuning LaBraM on the EEG dataset. It generates publishable figures based on averaged results across datasets, allowing to compare the general performance of one or more models.

## Setup

In [ ]:
from IPython.display import display
from pathlib import Path

import core.settings as settings
import core.utils as core_utils
import matplotlib.pyplot as plt
import os
import core.io as io_utils
from ipywidgets import widgets

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir('..')

def save_button(fig, save_dir: Path, default_name="untitled"):
    name_input = widgets.Text(value=default_name, description='Filename:', placeholder='untitled')

    save_button_comp = widgets.Button(description="Save Figure")
    output_comp = widgets.Output()
    def on_save_clicked(b):
        with output_comp:
            output_comp.clear_output()
            filename = name_input.value.strip() or default_name
            if not filename.endswith(".png"):
                filename += ".png"
            save_dir.mkdir(parents=True, exist_ok=True)

            try:
                save_path = save_dir / filename
                fig.savefig(save_path, dpi=300, bbox_inches='tight')
                print(f"Figure saved to: {save_path}")
            except Exception as e:
                print(f"Error saving figure: {e}")
    save_button_comp.on_click(on_save_clicked)
    display(widgets.VBox([widgets.HBox([name_input]), save_button_comp, output_comp]))



## Plotting training and validation losses across epochs

In [ ]:
model_name = "wavelet_120"

model_data = io_utils.load_model_log_data(settings.FINETUNE_OUTPUT_DIR, model_name=model_name)

avg_metrics = core_utils.avg_metrics_over_folds(model_data = model_data, metrics=["train_r2", "val_r2"])

avg_train = avg_metrics["train_r2"]
avg_val = avg_metrics["val_r2"]

# plot training and validation losses across epochs
fig = plt.figure(figsize=(10, 6))
plt.plot(avg_train, label="Train R2")
plt.plot(avg_val, label="Validation R2")
plt.xlabel("Epoch")
plt.ylabel("R2 Score")
plt.title(f"Training and Validation R2 Scores for {model_name}")
plt.legend()
plt.grid()
save_button(fig, save_dir=settings.PLOT_MACRO/"train_val", default_name=model_name)
plt.show()

## Plotting val performance of multiple models across epochs

In [ ]:
model_names = ["low_lr", "pre_low_lr", "wavelet_120", "temporal_high_embed", "temp_high_embed_no_pre"]

avg_vals = {}
for model_name in model_names:
    model_data = io_utils.load_model_log_data(settings.FINETUNE_OUTPUT_DIR, model_name=model_name)

    avg_metrics = core_utils.avg_metrics_over_folds(model_data = model_data, metrics=["val_r2"])
    avg_vals[model_name] = avg_metrics["val_r2"]

#plot validation losses across epochs on one line graph for all models
fig = plt.figure(figsize=(10, 6))
for model_name in model_names:
    if "pre" in model_name:
        label = f"{model_name} (pretrained)"
        linestyle = "--"
    else:
        label = model_name
        linestyle = "-"
    plt.plot(avg_vals[model_name], label=label, linestyle=linestyle)
plt.xlabel("Epoch")
plt.ylabel("Validation R2 Score")
plt.title("Validation R2 Scores Across Epochs for Multiple Models")
plt.legend()
plt.grid()
save_button(fig, save_dir=settings.PLOT_MACRO/"val_multiple")
plt.show()
